# YOLOv8 Fine-Tuning on DeepFashion In-Shop


## Cell 1 — Install Dependencies

In [1]:
!pip install ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.6 MB/s eta 0:00:00


In [2]:
!ls /kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset/Anno

attributes	      list_description_inshop.json  segmentation
densepose	      list_item_inshop.txt
list_bbox_inshop.txt  list_landmarks_inshop.txt


## Cell 2 — Imports & Paths
Set `DATASET_ROOT` to match your Kaggle dataset name.  
All outputs (labels, symlinks, weights) go to `/kaggle/working/`.


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
import shutil
from pathlib import Path

import pandas as pd
import yaml
from PIL import Image
from tqdm import tqdm

# ── ⚠️  Change 'your-dataset-name' to match your Kaggle dataset slug ──
DATASET_ROOT = Path("/kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset")
IMG_ROOT     = DATASET_ROOT / "img" / "img"   # images are nested at img/img/ on disk
BBOX_FILE    = DATASET_ROOT / "Anno" / "list_bbox_inshop.txt"
SPLIT_FILE   = DATASET_ROOT / "eval" / "list_eval_partition.txt"

# ── YOLO dataset output (writable Kaggle working dir) ──
YOLO_DIR    = Path("/kaggle/working/yolo_deepfashion")
YOLO_IMAGES = YOLO_DIR / "images"
YOLO_LABELS = YOLO_DIR / "labels"

# ── Training config ──
MODEL_NAME  = "yolov8l.pt"   # swap to yolov8n.pt if GPU RAM is tight
EPOCHS      = 30
IMG_SIZE    = 640
BATCH_SIZE  = 16
WORKERS     = 4
CLASS_NAMES = ["clothing"]   # single class — YOLO only needs to localise, not classify

print("Paths configured.")
print(f"  bbox  : {BBOX_FILE}")
print(f"  split : {SPLIT_FILE}")
print(f"  output: {YOLO_DIR}")


Paths configured.
  bbox  : /kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset/Anno/list_bbox_inshop.txt
  split : /kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset/eval/list_eval_partition.txt
  output: /kaggle/working/yolo_deepfashion


## Cell 3 — Parse Annotation Files

### `list_bbox_inshop.txt` format
```
52712                                          ← total count
image_name  clothes_type  pose_type  x_1  y_1  x_2  y_2   ← header
img/WOMEN/.../02_1_front.jpg  1  1  050  049  208  235     ← data
...
```

### `list_eval_partition.txt` format
```
52712
image_name  item_id  evaluation_status
img/WOMEN/.../02_1_front.jpg  id_00000002  train
...
```

> **Note:** paths in the txt files start with `img/` but images live at `img/img/` on disk, so we strip the leading `img/` prefix.


In [4]:
def parse_bbox_file(path: Path) -> pd.DataFrame:
    """
    Parses list_bbox_inshop.txt.
    Strips leading 'img/' from image paths so they resolve correctly under IMG_ROOT.
    """
    with open(path) as f:
        lines = f.readlines()
    rows = []
    for line in lines[2:]:          # skip count + header
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        raw_name = parts[0]
        img_name = raw_name[len("img/"):] if raw_name.startswith("img/") else raw_name
        rows.append(dict(
            image_name   = img_name,
            clothes_type = int(parts[1]),
            pose_type    = int(parts[2]),
            x1=int(parts[3]), y1=int(parts[4]),
            x2=int(parts[5]), y2=int(parts[6]),
        ))
    return pd.DataFrame(rows)


def parse_split_file(path: Path) -> pd.DataFrame:
    """
    Parses list_eval_partition.txt.
    evaluation_status ∈ {train, query, gallery}
    """
    with open(path) as f:
        lines = f.readlines()
    rows = []
    for line in lines[2:]:          # skip count + header
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        raw_name = parts[0]
        img_name = raw_name[len("img/"):] if raw_name.startswith("img/") else raw_name
        rows.append(dict(
            image_name = img_name,
            item_id    = parts[1],
            split      = parts[2],
        ))
    return pd.DataFrame(rows)


bbox_df  = parse_bbox_file(BBOX_FILE)
split_df = parse_split_file(SPLIT_FILE)

merged = bbox_df.merge(split_df, on="image_name", how="inner")

print(f"Total annotated images : {len(bbox_df)}")
print(f"After merge with split : {len(merged)}")
print(merged["split"].value_counts())
merged.head(3)


Total annotated images : 52712
After merge with split : 52712
split
train      25882
query      14218
gallery    12612
Name: count, dtype: int64


,image_name,clothes_type,pose_type,x1,y1,x2,y2,item_id,split
0,WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg,1,1,50,49,208,235,id_00000001,gallery
1,WOMEN/Blouses_Shirts/id_00000001/02_2_side.jpg,1,2,119,48,136,234,id_00000001,query
2,WOMEN/Blouses_Shirts/id_00000001/02_3_back.jpg,1,3,50,42,213,240,id_00000001,gallery


## Cell 4 — Convert to YOLO Label Format

YOLO expects one `.txt` label file per image with this format:
```
<class_id>  <x_center_norm>  <y_center_norm>  <width_norm>  <height_norm>
```
All coordinates are **normalised to [0, 1]** by the image dimensions.

DeepFashion gives **absolute pixel** coords `(x1, y1, x2, y2)` — we convert them here.

We use:
- **train** split → YOLO `train/`
- **query** split → YOLO `val/`  *(gallery is test-only, not needed for training)*

> Set `SUBSET_FRAC = 0.3` to use only 30% of training images if Kaggle disk/time is limited.


In [5]:
def abs_box_to_yolo(x1, y1, x2, y2, img_w, img_h):
    """Convert absolute pixel bbox to YOLO normalised format."""
    x_center = ((x1 + x2) / 2) / img_w
    y_center = ((y1 + y2) / 2) / img_h
    width    = (x2 - x1) / img_w
    height   = (y2 - y1) / img_h
    # Clamp to [0, 1] for safety
    x_center = min(max(x_center, 0.0), 1.0)
    y_center = min(max(y_center, 0.0), 1.0)
    width    = min(max(width,    0.0), 1.0)
    height   = min(max(height,   0.0), 1.0)
    return x_center, y_center, width, height


def build_yolo_dataset(df: pd.DataFrame, split_name: str, subset_frac: float = 1.0):
    """Write YOLO image symlinks and label .txt files for one split."""
    if subset_frac < 1.0:
        df = df.sample(frac=subset_frac, random_state=42).reset_index(drop=True)

    img_out_dir = YOLO_IMAGES / split_name
    lbl_out_dir = YOLO_LABELS / split_name
    img_out_dir.mkdir(parents=True, exist_ok=True)
    lbl_out_dir.mkdir(parents=True, exist_ok=True)

    skipped = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Building {split_name}"):
        src_img_path = IMG_ROOT / row["image_name"]
        if not src_img_path.exists():
            skipped += 1
            continue

        try:
            with Image.open(src_img_path) as im:
                img_w, img_h = im.size
        except Exception:
            skipped += 1
            continue

        x_c, y_c, w, h = abs_box_to_yolo(
            row["x1"], row["y1"], row["x2"], row["y2"], img_w, img_h
        )

        # Flat filename to avoid nested dirs (e.g. WOMEN_Dresses_id_xxx_01_1_front)
        flat_stem    = row["image_name"].replace("/", "_").replace(".jpg", "")
        lbl_path     = lbl_out_dir / f"{flat_stem}.txt"
        dst_img_path = img_out_dir  / f"{flat_stem}.jpg"

        with open(lbl_path, "w") as f:
            f.write(f"0 {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

        if not dst_img_path.exists():
            try:
                dst_img_path.symlink_to(src_img_path.resolve())
            except Exception:
                shutil.copy2(src_img_path, dst_img_path)

    print(f"  {split_name}: {len(df) - skipped} images ready, {skipped} skipped.")


train_df = merged[merged["split"] == "train"]
val_df   = merged[merged["split"] == "query"]

SUBSET_FRAC = 1.0   # ← set to 0.3 or 0.5 to speed up

build_yolo_dataset(train_df, "train", subset_frac=SUBSET_FRAC)
build_yolo_dataset(val_df,   "val",   subset_frac=1.0)


Building train: 100%|██████████| 25882/25882 [05:35<00:00, 77.11it/s]


  train: 25882 images ready, 0 skipped.


Building val: 100%|██████████| 14218/14218 [03:06<00:00, 76.41it/s]

  val: 14218 images ready, 0 skipped.


## Cell 5 — Write `data.yaml`
YOLO needs a YAML config pointing to the image directories and class names.


In [6]:
data_yaml = {
    "path"  : str(YOLO_DIR),
    "train" : "images/train",
    "val"   : "images/val",
    "nc"    : 1,
    "names" : CLASS_NAMES,
}

yaml_path = YOLO_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("data.yaml written:")
print(open(yaml_path).read())


data.yaml written:
names:
- clothing
nc: 1
path: /kaggle/working/yolo_deepfashion
train: images/train
val: images/val



## Cell 6 — Fine-Tune YOLOv8

**Key training choices:**
| Setting | Value | Reason |
|---|---|---|
| Model | `yolov8s.pt` | Good balance of speed vs accuracy; swap to `n` if OOM |
| Epochs | 30 | With early stopping (patience=10), usually converges earlier |
| Optimizer | AdamW | More stable than SGD for fine-tuning |
| `fliplr=0.5` | horizontal flip | Clothes are left-right symmetric |
| `flipud=0.0` | no vertical flip | Clothes are not upside-down symmetric |
| `mosaic=0.5` | mosaic aug | Helps with small/partial clothing items |


In [7]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)   # downloads pretrained weights automatically

results = model.train(
    data         = str(yaml_path),
    epochs       = EPOCHS,
    imgsz        = IMG_SIZE,
    batch        = BATCH_SIZE,
    workers      = WORKERS,
    project      = "/kaggle/working/runs",
    name         = "yolo_deepfashion",
    exist_ok     = True,
    device = "0,1",
    # ── Augmentations ──
    hsv_h        = 0.015,   # hue jitter
    hsv_s        = 0.5,     # saturation jitter
    hsv_v        = 0.3,     # brightness jitter
    fliplr       = 0.5,     # horizontal flip
    flipud       = 0.0,     # no vertical flip
    scale        = 0.3,     # zoom ±30%
    translate    = 0.1,     # translate ±10%
    mosaic       = 0.5,     # mosaic probability

    # ── Optimizer ──
    optimizer    = "AdamW",
    lr0          = 1e-3,
    lrf          = 0.01,
    weight_decay = 5e-4,

    # ── Early stopping & checkpointing ──
    patience     = 10,
    save         = True,
    save_period  = 5,       # checkpoint every 5 epochs
)

print("Training complete.")
print(f"Best weights: {results.save_dir}/weights/best.pt")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_deepfashion/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, f

AttributeError: 'NoneType' object has no attribute 'save_dir'

## Cell 7 — Validate on Val Split
Run official YOLO validation on the query split to get detection metrics.


In [ ]:
best_weights = Path(results.save_dir) / "weights" / "best.pt"
model_best   = YOLO(str(best_weights))

val_metrics = model_best.val(
    data    = str(yaml_path),
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    workers = WORKERS,
)

print("\n── Validation Metrics ──")
print(f"  mAP@0.5      : {val_metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95 : {val_metrics.box.map:.4f}")
print(f"  Precision    : {val_metrics.box.mp:.4f}")
print(f"  Recall       : {val_metrics.box.mr:.4f}")


## Cell 8 — Visual Smoke Test
Randomly sample 6 images from the val split and visualise predicted bounding boxes.


In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def visualise_predictions(model, img_paths: list, n: int = 6):
    sample = random.sample(img_paths, min(n, len(img_paths)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for ax, img_path in zip(axes, sample):
        full_path = IMG_ROOT / img_path
        pil_img   = Image.open(full_path).convert("RGB")
        result    = model.predict(source=np.array(pil_img), conf=0.25, verbose=False)[0]

        ax.imshow(pil_img)
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor="red", facecolor="none"
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 5, f"{conf:.0%}", color="red", fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("/kaggle/working/sample_predictions.png", dpi=100)
    plt.show()
    print("Saved → /kaggle/working/sample_predictions.png")


sample_paths = val_df["image_name"].sample(6, random_state=0).tolist()
visualise_predictions(model_best, sample_paths)


## Cell 9 — Export Best Weights
Copy `best.pt` to the Kaggle output tab so you can download it.  
Drop the downloaded file next to `yolo_localizer.py` and initialise with:
```python
localizer = YOLOLocalizer(model_path="yolo_deepfashion_best.pt")
```


In [ ]:
output_weights = Path("/kaggle/working/yolo_deepfashion_best.pt")
shutil.copy2(best_weights, output_weights)
print(f"✅ Download from Kaggle output tab: {output_weights}")
